IMPORS DE LIBRERIAS A UTILIZAR 

In [39]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time 
import sqlite3

CHECKEO PREVIO

In [40]:
try:
    url = 'https://books.toscrape.com/'
    respuesta = requests.get(url)
    if respuesta.status_code == 200:
        print('conexion exitosa') 
except (Exception, KeyboardInterrupt) as e:
    print(f"conexion fallida {e}")    

conexion exitosa


ITERACION DE LAS 50 PAGINAS 

In [41]:
libro_datos = []  # Creamos una lista que almacenará los datos de los libros
for page_num in range(1, 51):
    paginas = f'https://books.toscrape.com/catalogue/page-{page_num}.html'
    response = requests.get(paginas)

    if response.status_code != 200:
        print(f"Error en página {page_num}: HTTP {response.status_code}")
        continue  
    
    #response.content trae la informacion en bytes el encoding lo decide beatifulsoup
    soup = BeautifulSoup(response.content, 'html.parser') 
    libros = soup.find_all('h3')

    for libro in libros:
        try:
            # Encontrar la URL vinculada al libro (libro.find)
            libro_url = libro.find('a')['href']
            libro_response = requests.get('https://books.toscrape.com/catalogue/' + libro_url)
            if libro_response.status_code != 200:
               raise Exception(f"Error en libro: HTTP {libro_response.status_code}")  # salta al siguiente libro
            libro_soup = BeautifulSoup(libro_response.content, "html.parser")

            # Encontrar el título del libro mediante la etiqueta del h1
            titulo = libro_soup.find('h1').text
            categoria = libro_soup.find('ul', class_="breadcrumb").find_all('a')[2].text.strip()
            calificacion = libro_soup.find('p', class_='star-rating')['class'][1]
            precio = libro_soup.find('p', class_="price_color").text.strip()
            libro_datos.append([titulo, categoria, calificacion, precio])
            print(f"Titulo: {titulo}")
            print(f"Categoria: {categoria}")
            print(f"Calificacion: {calificacion}")
            print(f"Precio: {precio}")
        except (Exception,KeyboardInterrupt) as e:
            print(f"Hubo un error {e}")
            continue

Titulo: A Light in the Attic
Categoria: Poetry
Calificacion: Three
Precio: £51.77
Titulo: Tipping the Velvet
Categoria: Historical Fiction
Calificacion: One
Precio: £53.74
Titulo: Soumission
Categoria: Fiction
Calificacion: One
Precio: £50.10
Titulo: Sharp Objects
Categoria: Mystery
Calificacion: Four
Precio: £47.82
Titulo: Sapiens: A Brief History of Humankind
Categoria: History
Calificacion: Five
Precio: £54.23
Titulo: The Requiem Red
Categoria: Young Adult
Calificacion: One
Precio: £22.65
Titulo: The Dirty Little Secrets of Getting Your Dream Job
Categoria: Business
Calificacion: Four
Precio: £33.34
Titulo: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Categoria: Default
Calificacion: Three
Precio: £17.93
Titulo: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
Categoria: Default
Calificacion: Four
Precio: £22.60
Titulo: The Black Maria
Categoria: Poetry
Calificacion: One
Precio: £52.15
Titulo: S

CONVERSION DE LISTAS A CSV CON PANDA

In [ ]:
dataframe = pd.DataFrame(libro_datos, columns=["titulo", "categoria", "calificacion", "precio"])

dataframe["precio"] = dataframe["precio"].str.replace("£", "").astype(float)
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
dataframe["calificacion"] = dataframe["calificacion"].map(rating_map)

In [43]:
# Agregar id único por fila para relacionar con los demás CSVs
dataframe["autor_id"] = range(len(dataframe))
# index=False evita que pandas agregue una columna extra con el índice numérico al CSV
dataframe.to_csv("data/libros_scrapeados.csv", index=False)
dataframe = pd.read_csv("data/libros_scrapeados.csv")

dataframe

,titulo,categoria,calificacion,precio,autor_id
0,A Light in the Attic,Poetry,3,51.77,0
1,Tipping the Velvet,Historical Fiction,1,53.74,1
2,Soumission,Fiction,1,50.10,2
3,Sharp Objects,Mystery,4,47.82,3
4,Sapiens: A Brief History of Humankind,History,5,54.23,4
...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,Classics,1,55.53,995
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",Sequential Art,4,57.06,996
997,A Spy's Devotion (The Regency Spies of London #1),Historical Fiction,5,16.97,997
998,1st to Die (Women's Murder Club #1),Mystery,1,53.98,998


OBTENER LOS KEYS DE LOS AUTORES CON EL TITULO DE SU OBRA (OPEN-LIBRARY)

In [44]:
datos = pd.read_csv("data/libros_scrapeados.csv")
titulos = datos['titulo'].tolist()
claves = []
for titulo in titulos:
    try:
        titulo_corto = titulo.split(":")[0].split("(")[0].split(",")[0].strip()
        params = {"title": titulo_corto, "limit": 1}
        url_opl = "https://openlibrary.org/search.json"
        response = requests.get(url_opl, params=params, timeout=10)
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}")
        data = response.json()
        doc = data["docs"][0]
        opl_id = doc["author_key"][0]

        claves.append([opl_id])
        print(claves)
    except (Exception,KeyboardInterrupt) as e:
        print(f"No encontrado: {titulo} - {e}")
        claves.append([None])
        continue



[['OL548174A']]
[['OL548174A'], ['OL39232A']]
[['OL548174A'], ['OL39232A'], ['OL300477A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], ['OL7607124A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], ['OL7607124A'], ['OL1168086A']]
No encontrado: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics - HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=10)
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], ['OL7607124A'], ['OL1168086A'], [None], ['OL1391804A']]
[['OL548174A'], ['OL39232A'], ['OL300477A'], ['OL1433006A'], ['OL3778242A'], ['OL3905230A'], [

CONVERSION DE LISTAS DE AUTOR_KEYS A CSV CON PANDA

In [45]:
dataframe = pd.DataFrame(claves, columns=["autor_key"])

dataframe["autor_id"] = range(len(dataframe))

dataframe.to_csv("data/autor_key.csv", index=False)
dataframe

,autor_key,autor_id
0,OL548174A,0
1,OL39232A,1
2,OL300477A,2
3,OL1433006A,3
4,OL3778242A,4
...,...,...
995,OL22098A,995
996,OL7423510A,996
997,OL7032059A,997
998,OL22258A,998


In [46]:
df_keys = pd.read_csv("data/autor_key.csv")
df_keys["autor_id"] = range(len(df_keys))
df_keys.to_csv("data/autor_key.csv", index=False)
df_keys

,autor_key,autor_id
0,OL548174A,0
1,OL39232A,1
2,OL300477A,2
3,OL1433006A,3
4,OL3778242A,4
...,...,...
995,OL22098A,995
996,OL7423510A,996
997,OL7032059A,997
998,OL22258A,998


ENRIQUECIMIENTO DE DATOS DE AUTOR MEDIANTE AUTOR_KEY (OPEN-LIBRARY)

In [47]:
dato_key = pd.read_csv("data/autor_key.csv")
autor_keys = dato_key['autor_key'].tolist()

autor_datos = []

for key in autor_keys:
    if pd.isna(key):
        autor_datos.append([None, None, None, None])
        continue
    try:
        url_autor = f"https://openlibrary.org/authors/{key}.json"
        url_works = f"https://openlibrary.org/authors/{key}/works.json"

        response_autor = requests.get(url_autor, timeout=10)
        if response_autor.status_code != 200:
            raise Exception(f"HTTP {response_autor.status_code}")
        response_works = requests.get(url_works, timeout=10)
    
        if response_works.status_code != 200:
            raise Exception(f"HTTP {response_works.status_code}")
        data_works = response_works.json()
        data_autor = response_autor.json()

        fecha_nacimiento = data_autor.get("birth_date", None)
        nombre = data_autor.get("personal_name", None)
        fecha_creacion = data_autor.get("created", {}).get("value", None)
        total_obras = data_works.get("size", None)

        ano_nacimiento = None
        if fecha_nacimiento:
            for i in range(len(fecha_nacimiento) - 3):
                parte = fecha_nacimiento[i:i+4]
                if parte.isdigit():
                    ano_nacimiento = int(parte)
                    break
        autor_datos.append([ano_nacimiento, nombre, fecha_creacion, total_obras])
        time.sleep(0.5) 
        print(f"Nombre: {nombre}")
        print(f"Año: {ano_nacimiento}")
        print(f"Creación: {fecha_creacion}")
        print(f"Cantidad Obras: {total_obras}")
    except (Exception,KeyboardInterrupt) as e:
        print(f"Error: {key} - {e}")
        autor_datos.append([None, None, None, None])
        continue



Nombre: Shel Silverstein
Año: None
Creación: 2008-04-01T03:28:50.625462
Cantidad Obras: 152
Nombre: Sarah Waters
Año: 1966
Creación: 2008-04-01T03:28:50.625462
Cantidad Obras: 36
Nombre: Michel Houellebecq
Año: 1956
Creación: 2008-04-01T03:28:50.625462
Cantidad Obras: 85
Nombre: Gillian Flynn
Año: 1971
Creación: 2008-04-01T03:28:50.625462
Cantidad Obras: 43
Nombre: Yuval N. Harari
Año: 1976
Creación: 2008-04-30T20:50:18.033121
Cantidad Obras: 52
Nombre: None
Año: None
Creación: None
Cantidad Obras: 8
Error: OL7607124A - HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /authors/OL7607124A.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='openlibrary.org', port=443) at 0x1b6621a4410>, 'Connection to openlibrary.org timed out. (connect timeout=10)'))
Nombre: Ariana Wormeley Curtis
Año: 1833
Creación: 2008-04-01T03:28:50.625462
Cantidad Obras: 8
Nombre: Aracelis Girmay
Año: None
Creación: 2008-04-01T03:28:50.625462
Cantidad Obras: 13
Nombre: No

CONVERSION LISTA DE DATOS DE AUTOR A CSV MEDIANTE PANDAS

In [48]:
dataframe = pd.DataFrame(autor_datos, columns=["ano_nacimiento", "nombre", "fecha_creacion", "total_obras"])

dataframe.to_csv("data/autor_datos.csv", index=False)
dataframe

,ano_nacimiento,nombre,fecha_creacion,total_obras
0,NaN,Shel Silverstein,2008-04-01T03:28:50.625462,152.0
1,1966.0,Sarah Waters,2008-04-01T03:28:50.625462,36.0
2,1956.0,Michel Houellebecq,2008-04-01T03:28:50.625462,85.0
3,1971.0,Gillian Flynn,2008-04-01T03:28:50.625462,43.0
4,1976.0,Yuval N. Harari,2008-04-30T20:50:18.033121,52.0
...,...,...,...,...
995,1832.0,"Carroll, Lewis",2008-04-01T03:28:50.625462,2654.0
996,NaN,NaN,NaN,NaN
997,NaN,NaN,NaN,NaN
998,1947.0,"Patterson, James",2008-04-01T03:28:50.625462,743.0


In [49]:
dataframe = pd.read_csv("data/autor_datos.csv")
dataframe["ano_nacimiento"] = pd.to_numeric(dataframe["ano_nacimiento"], errors="coerce").astype("Int64")
dataframe["total_obras"] = pd.to_numeric(dataframe["total_obras"], errors="coerce").astype("Int64")

dataframe["autor_id"] = range(len(dataframe))
dataframe.to_csv("data/autor_datos.csv", index=False)
dataframe

,ano_nacimiento,nombre,fecha_creacion,total_obras,autor_id
0,<NA>,Shel Silverstein,2008-04-01T03:28:50.625462,152,0
1,1966,Sarah Waters,2008-04-01T03:28:50.625462,36,1
2,1956,Michel Houellebecq,2008-04-01T03:28:50.625462,85,2
3,1971,Gillian Flynn,2008-04-01T03:28:50.625462,43,3
4,1976,Yuval N. Harari,2008-04-30T20:50:18.033121,52,4
...,...,...,...,...,...
995,1832,"Carroll, Lewis",2008-04-01T03:28:50.625462,2654,995
996,<NA>,NaN,NaN,<NA>,996
997,<NA>,NaN,NaN,<NA>,997
998,1947,"Patterson, James",2008-04-01T03:28:50.625462,743,998


ENRIQUECIMIENTO DE NACIONALIDAD DE AUTORES MEDIANTE NOMBRE DE AUTOR (WIKIPEDIA)

In [50]:
dato_nombre = pd.read_csv("data/autor_datos.csv")
autor_nombre = dato_nombre['nombre'].tolist()

nacionalidades = []
for nombre in autor_nombre:
    if pd.isna(nombre):
        nacionalidades.append(None)
        continue
    try:
        headers = {
            "User-Agent": "books-scrape-challenge/1.0 (hugo@email.com)"
        }
        nombre_limpio = nombre.strip().rstrip(".")
        url_wiki = f"https://en.wikipedia.org/api/rest_v1/page/summary/{nombre_limpio.replace(' ', '_')}"
        response_wiki = requests.get(url_wiki, headers=headers, timeout=10)
        if response_wiki.status_code != 200:
            raise Exception(f"HTTP {response_wiki.status_code}")
        data_wiki = response_wiki.json()
        descripcion = data_wiki.get("description", None)

        if descripcion and descripcion != "Topics referred to by the same term":
            nacionalidad = descripcion.split()[0]
        else:
            nacionalidad = None
        nacionalidades.append(nacionalidad)
        time.sleep(0.3)
        print(nacionalidades)
    except (Exception, KeyboardInterrupt) as e:
        print(f"Error: {nombre} - {e}")
        nacionalidades.append(None)
        continue

['American']
['American', 'Welsh']
['American', 'Welsh', 'French']
['American', 'Welsh', 'French', 'American']
['American', 'Welsh', 'French', 'American', 'Israeli']
Error: Ariana Wormeley Curtis - HTTP 404
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American']
Error: James Schiffer - HTTP 404
Error: Robins, Denise - HTTP 404
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', None, None, None, None, None, 'American']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', None, None, None, None, None, 'American', 'American']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', None, None, None, None, None, 'American', 'American', None, None, None, None, 'American']
['American', 'Welsh', 'French', 'American', 'Israeli', None, None, None, None, 'American', None, None, None, None, None, 'American', 'American', None, None, None, None, 'America

CONVERSION LISTA DE NACIONALIDADES A CSV MEDIANTE PANDAS

In [51]:
dataframe = pd.DataFrame(nacionalidades, columns=["nacionalidad"])

dataframe.to_csv("data/autor_nacionalidad.csv", index=False)
dataframe

,nacionalidad
0,American
1,Welsh
2,French
3,American
4,Israeli
...,...
995,British
996,NaN
997,NaN
998,American


In [52]:
dataframe = pd.read_csv("data/autor_nacionalidad.csv")
dataframe["autor_id"] = range(len(dataframe))
dataframe.to_csv("data/autor_nacionalidad.csv", index=False)
dataframe

,nacionalidad,autor_id
0,American,0
1,Welsh,1
2,French,2
3,American,3
4,Israeli,4
...,...,...
995,British,995
996,NaN,996
997,NaN,997
998,American,998


VERIFICAMOS CANTIDAD DE INFORMACION ALMACENADA EN LOS CSV

In [53]:
# Leer los CSVs a través de la ruta de archivos
df_libros = pd.read_csv("data/libros_scrapeados.csv")
df_keys = pd.read_csv("data/autor_key.csv")
df_datos = pd.read_csv("data/autor_datos.csv")
df_nacion = pd.read_csv("data/autor_nacionalidad.csv")
# Imprimimos la cantidad de filas de cada CSV para ver cantidad de información
print(f"libros: {len(df_libros)}, claves: {len(df_keys)},autores: {len(df_datos)}, nacionalidad: {len(df_nacion)}")

libros: 1000, claves: 1000,autores: 1000, nacionalidad: 1000


In [54]:
dataframe = pd.read_csv("data/autor_datos.csv")
print(dataframe["fecha_creacion"].head(10))

0    2008-04-01T03:28:50.625462
1    2008-04-01T03:28:50.625462
2    2008-04-01T03:28:50.625462
3    2008-04-01T03:28:50.625462
4    2008-04-30T20:50:18.033121
5                           NaN
6                           NaN
7    2008-04-01T03:28:50.625462
8                           NaN
9    2008-04-01T03:28:50.625462
Name: fecha_creacion, dtype: str


CONEXION CON LA BASE DE DATOS

In [55]:
conn = sqlite3.connect("Libros.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON")
print('conexion exitosa')

conexion exitosa


CREAR TABLAS

In [56]:
esquema = """
DROP TABLE IF EXISTS libros_autores;
DROP TABLE IF EXISTS libros;
DROP TABLE IF EXISTS autores;
DROP TABLE IF EXISTS categorias;

CREATE TABLE IF NOT EXISTS categorias(
    categoria_id INTEGER PRIMARY KEY,
    categoria_name TEXT
);

CREATE TABLE IF NOT EXISTS autores(
    autor_id     INTEGER PRIMARY KEY,
    autor_key    TEXT,
    nombre       TEXT,
    ano_nacimiento INTEGER,
    nacionalidad TEXT,
    total_obras  INTEGER,
    fecha_creacion TEXT
);

CREATE TABLE IF NOT EXISTS libros(
    libro_id     INTEGER PRIMARY KEY,
    titulo       TEXT,
    precio       NUMERIC,
    calificacion INTEGER,
    categoria_id INTEGER REFERENCES categorias(categoria_id)
);

CREATE TABLE IF NOT EXISTS libros_autores(
    libro_id  INTEGER NOT NULL REFERENCES libros(libro_id),
    autor_id  INTEGER NOT NULL REFERENCES autores(autor_id),
    PRIMARY KEY (libro_id, autor_id)
);
"""
cursor.executescript(esquema)
conn.commit()
print("Tablas creadas")


Tablas creadas


***PREPARAR LOS DATOS ANTES DE to_sql**

TABLA CATEGORIA

In [57]:
# Extraer categorías únicas y asignar categoria_id desde 1
df_categorias = pd.DataFrame(df_libros["categoria"].unique(), columns=["categoria_name"])
df_categorias.index += 1  # index arranca en 1, se convierte en categoria_id

# Mapa categoria_name → categoria_id para usarlo en libros
cat_map = dict(zip(df_categorias["categoria_name"], df_categorias.index))
df_libros["categoria_id"] = df_libros["categoria"].map(cat_map)

print("Datos Preparados")


Datos Preparados


TABLA AUTORES

In [58]:
df_autores = (df_keys.merge(df_datos, on="autor_id", how="left")
                     .merge(df_nacion, on="autor_id", how="left"))

df_autores["ano_nacimiento"] = pd.to_numeric(df_autores["ano_nacimiento"], errors="coerce").astype("Int64")
df_autores["total_obras"]    = pd.to_numeric(df_autores["total_obras"],    errors="coerce").astype("Int64")

# autor_id arranca en 1
df_autores["autor_id"] = range(1, len(df_autores) + 1)

# Seleccionar solo columnas que existen en la tabla autores
df_autores = df_autores[["autor_id", "autor_key", "nombre", "ano_nacimiento",
                          "nacionalidad", "total_obras", "fecha_creacion"]]
print("Datos Preparados")
print(df_autores.head())


Datos Preparados
   autor_id   autor_key              nombre  ano_nacimiento nacionalidad  \
0         1   OL548174A    Shel Silverstein            <NA>     American   
1         2    OL39232A        Sarah Waters            1966        Welsh   
2         3   OL300477A  Michel Houellebecq            1956       French   
3         4  OL1433006A       Gillian Flynn            1971     American   
4         5  OL3778242A     Yuval N. Harari            1976      Israeli   

   total_obras              fecha_creacion  
0          152  2008-04-01T03:28:50.625462  
1           36  2008-04-01T03:28:50.625462  
2           85  2008-04-01T03:28:50.625462  
3           43  2008-04-01T03:28:50.625462  
4           52  2008-04-30T20:50:18.033121  


TABLA LIBRO_AUTORES

In [59]:
# libro_id  = posición del libro (1..1000), alineado con el rowid que to_sql asigna
# autor_id  = autor_id del libro, que ya arranca en 1
df_libros_autores = pd.DataFrame({
    "libro_id":  range(1, len(df_libros) + 1),
    "autor_id":  df_autores["autor_id"].values
})
print("Datos Preparados")
print(df_libros_autores.head())


Datos Preparados
   libro_id  autor_id
0         1         1
1         2         2
2         3         3
3         4         4
4         5         5


***CARGA DE DATOS CON to_sql**

In [60]:
# categorias: index=True lo convierte en categoria_id (arranca en 1)
df_categorias.to_sql("categorias", conn, if_exists="replace",
                     index=True, index_label="categoria_id")

# libros: seleccionamos solo las columnas del schema
df_libros[["titulo", "precio", "calificacion", "categoria_id"]].to_sql(
    "libros", conn, if_exists="replace", index=True, index_label="libro_id")

# autores: index=False porque autor_id ya está en el DataFrame
df_autores.to_sql("autores", conn, if_exists="replace", index=False)

# libros_autores: tabla pivote sin índice extra
df_libros_autores.to_sql("libros_autores", conn, if_exists="replace", index=False)

conn.commit()
print("Carga completada")


Carga completada


***INDICES**

In [62]:

cursor.executescript("""
    -- FK joins: los más importantes, aceleran todos los JOINs
    CREATE INDEX IF NOT EXISTS idx_libros_categoria_id
        ON libros(categoria_id);

    CREATE INDEX IF NOT EXISTS idx_libros_autores_libro
        ON libros_autores(libro_id);

    CREATE INDEX IF NOT EXISTS idx_libros_autores_autor
        ON libros_autores(autor_id);

    -- Filtros WHERE frecuentes
    CREATE INDEX IF NOT EXISTS idx_libros_calificacion
        ON libros(calificacion);

    CREATE INDEX IF NOT EXISTS idx_libros_precio
        ON libros(precio);

    -- Índice compuesto: WHERE calificacion > 3 AND precio < X
    -- SQLite usa este en vez de los dos anteriores cuando aparecen juntos
    CREATE INDEX IF NOT EXISTS idx_libros_calif_precio
        ON libros(calificacion, precio);

    -- Búsquedas y GROUP BY por país
    CREATE INDEX IF NOT EXISTS idx_autores_nacionalidad
        ON autores(nacionalidad);
""")

conn.commit()

# Verificar que se crearon
cursor.execute("""
    SELECT name, tbl_name
    FROM sqlite_master
    WHERE type = 'index'
    ORDER BY tbl_name, name
""")

print(f"{'Tabla':<20} {'Índice'}")
print("─" * 50)
for tabla, indice in [(r[1], r[0]) for r in cursor.fetchall()]:
    print(f"{tabla:<20} {indice}")

conn.close()
print("\n✓ Índices creados")

Tabla                Índice
──────────────────────────────────────────────────
autores              idx_autores_nacionalidad
categorias           ix_categorias_categoria_id
libros               idx_libros_calif_precio
libros               idx_libros_calificacion
libros               idx_libros_categoria_id
libros               idx_libros_precio
libros               ix_libros_libro_id
libros_autores       idx_libros_autores_autor
libros_autores       idx_libros_autores_libro

✓ Índices creados


CONSULTAS 

In [ ]:


consultas = {
    "1. Libros baratos bien rankeados (calificacion > 3, precio < £20)": """
        SELECT l.titulo, l.precio, l.calificacion, c.categoria_name
        FROM libros l
        JOIN categorias c ON l.categoria_id = c.categoria_id
        WHERE l.calificacion > 3
          AND l.precio < 20
        ORDER BY l.calificacion DESC, l.precio ASC
        LIMIT 10
    """,

    "2. Peor promedio de calificacion (minimo 5 libros)": """
        SELECT a.nombre, a.nacionalidad,
               COUNT(l.libro_id)            AS total_libros,
               ROUND(AVG(l.calificacion),2) AS promedio
        FROM autores a
        JOIN libros_autores la ON la.autor_id = a.autor_id
        JOIN libros l          ON la.libro_id = l.libro_id
        WHERE a.nombre IS NOT NULL
        GROUP BY a.autor_id
        HAVING COUNT(l.libro_id) >= 5
        ORDER BY promedio ASC
        LIMIT 10
    """,

    "3. Categoria con mayor precio promedio": """
        SELECT c.categoria_name,
               COUNT(l.libro_id)        AS total_libros,
               ROUND(AVG(l.precio),2)   AS precio_promedio,
               ROUND(MIN(l.precio),2)   AS precio_minimo,
               ROUND(MAX(l.precio),2)   AS precio_maximo
        FROM categorias c
        JOIN libros l ON l.categoria_id = c.categoria_id
        GROUP BY c.categoria_id
        ORDER BY precio_promedio DESC
        LIMIT 10
    """,

    "4. Top 5 autores con mas libros": """
        SELECT a.nombre, a.nacionalidad,
               a.total_obras                AS obras_totales_ol,
               COUNT(l.libro_id)            AS libros_en_catalogo,
               ROUND(AVG(l.calificacion),2) AS promedio
        FROM autores a
        JOIN libros_autores la ON la.autor_id = a.autor_id
        JOIN libros l          ON la.libro_id = l.libro_id
        WHERE a.nombre IS NOT NULL
        GROUP BY a.autor_id
        ORDER BY libros_en_catalogo DESC
        LIMIT 5
    """,

    "5. Pais con mas libros con calificacion > 3": """
        SELECT a.nacionalidad              AS pais,
               COUNT(l.libro_id)          AS libros_rating_alto,
               ROUND(AVG(l.calificacion),2) AS promedio,
               ROUND(AVG(l.precio),2)     AS precio_promedio
        FROM libros l
        JOIN libros_autores la ON la.libro_id = l.libro_id
        JOIN autores a         ON la.autor_id = a.autor_id
        WHERE l.calificacion > 3
          AND a.nacionalidad IS NOT NULL
        GROUP BY a.nacionalidad
        ORDER BY libros_rating_alto DESC
        LIMIT 10
    """,
}

conn = sqlite3.connect("Libros.db")
for nombre, sql in consultas.items():
    print(f"\n{'='*60}")
    print(f" {nombre}")
    print(f"{'='*60}")
    df = pd.read_sql_query(sql, conn)
    print(df.to_string(index=False))

conn.close()



 1. Libros baratos bien rankeados (calificacion > 3, precio < £20)


ProgrammingError: Cannot operate on a closed database.

BENCHMARK — CONSULTA LENTA vs CON ÍNDICE

In [ ]:

# Borramos todos los índices creados antes
cursor.executescript("""
    DROP INDEX IF EXISTS idx_libros_categoria_id;
    DROP INDEX IF EXISTS idx_libros_autores_libro;
    DROP INDEX IF EXISTS idx_libros_autores_autor;
    DROP INDEX IF EXISTS idx_libros_calificacion;
    DROP INDEX IF EXISTS idx_libros_precio;
    DROP INDEX IF EXISTS idx_libros_calif_precio;
    DROP INDEX IF EXISTS idx_autores_nacionalidad;
""")
conn.commit()
print("Índices eliminados — tabla sin optimización")

QUERY = """
    SELECT l.titulo, l.precio, l.calificacion,
           c.categoria_name, a.nombre, a.nacionalidad
    FROM libros l
    JOIN categorias c      ON l.categoria_id = c.categoria_id
    JOIN libros_autores la ON la.libro_id     = l.libro_id
    JOIN autores a         ON la.autor_id     = a.autor_id
    WHERE l.calificacion > 3
      AND l.precio < 20
      AND a.nacionalidad IS NOT NULL
    ORDER BY l.calificacion DESC, l.precio ASC
"""

REPETICIONES = 200
inicio = time.perf_counter()
for _ in range(REPETICIONES):
    cursor.execute(QUERY)
    cursor.fetchall()
fin = time.perf_counter()

tiempo_sin_indice = (fin - inicio) / REPETICIONES * 1000  # ms por ejecución
print(f"\nSIN índice  →  {tiempo_sin_indice:.4f} ms por consulta")
print(f"({REPETICIONES} repeticiones, total {fin - inicio:.3f} s)")


In [ ]:

cursor.executescript("""
    CREATE INDEX IF NOT EXISTS idx_libros_categoria_id
        ON libros(categoria_id);

    CREATE INDEX IF NOT EXISTS idx_libros_autores_libro
        ON libros_autores(libro_id);

    CREATE INDEX IF NOT EXISTS idx_libros_autores_autor
        ON libros_autores(autor_id);

    CREATE INDEX IF NOT EXISTS idx_libros_calificacion
        ON libros(calificacion);

    CREATE INDEX IF NOT EXISTS idx_libros_precio
        ON libros(precio);

    CREATE INDEX IF NOT EXISTS idx_libros_calif_precio
        ON libros(calificacion, precio);

    CREATE INDEX IF NOT EXISTS idx_autores_nacionalidad
        ON autores(nacionalidad);
""")
conn.commit()
print("Índices recreados")

# PASO 4 — Medir tiempo CON índice
inicio = time.perf_counter()
for _ in range(REPETICIONES):
    cursor.execute(QUERY)
    cursor.fetchall()
fin = time.perf_counter()

tiempo_con_indice = (fin - inicio) / REPETICIONES * 1000
print(f"\nCON índice  →  {tiempo_con_indice:.4f} ms por consulta")
print(f"({REPETICIONES} repeticiones, total {fin - inicio:.3f} s)")


In [ ]:
mejora = tiempo_sin_indice / tiempo_con_indice

print("=" * 50)
print(" RESULTADO DEL BENCHMARK")
print("=" * 50)
print(f"  Sin índice  : {tiempo_sin_indice:.4f} ms")
print(f"  Con índice  : {tiempo_con_indice:.4f} ms")
print(f"  Mejora      : {mejora:.1f}x más rápido")
print("=" * 50)

# ── EXPLAIN: muestra qué estrategia usa SQLite en cada caso ──
print("\nEXPLAIN QUERY PLAN (con índice):")
cursor.execute("EXPLAIN QUERY PLAN " + QUERY)
for row in cursor.fetchall():
    print(" ", row)

# ── Resumen legible en DataFrame ──
df_benchmark = pd.DataFrame({
    "Escenario":     ["Sin índice", "Con índice"],
    "Tiempo (ms)":   [round(tiempo_sin_indice, 4), round(tiempo_con_indice, 4)],
    "Mejora":        ["-", f"{mejora:.1f}x"]
})
print("\n")
print(df_benchmark.to_string(index=False))

conn.close()
print("\n✓ Benchmark completado")
mejora = tiempo_sin_indice / tiempo_con_indice

print("=" * 50)
print(" RESULTADO DEL BENCHMARK")
print("=" * 50)
print(f"  Sin índice  : {tiempo_sin_indice:.4f} ms")
print(f"  Con índice  : {tiempo_con_indice:.4f} ms")
print(f"  Mejora      : {mejora:.1f}x más rápido")
print("=" * 50)

# ── EXPLAIN: muestra qué estrategia usa SQLite en cada caso ──
print("\nEXPLAIN QUERY PLAN (con índice):")
cursor.execute("EXPLAIN QUERY PLAN " + QUERY)
for row in cursor.fetchall():
    print(" ", row)

# ── Resumen legible en DataFrame ──
df_benchmark = pd.DataFrame({
    "Escenario":     ["Sin índice", "Con índice"],
    "Tiempo (ms)":   [round(tiempo_sin_indice, 4), round(tiempo_con_indice, 4)],
    "Mejora":        ["-", f"{mejora:.1f}x"]
})
print("\n")
print(df_benchmark.to_string(index=False))

conn.close()
print("\n✓ Benchmark completado")
